# Deep Learning 024 — Dropout, Part 1: Theory

Companion notebook to the lesson. Dropout switches off a random subset of neurons on every
forward pass. That sounds like sabotage, and the two standard explanations for why it helps
— *it is an implicit ensemble* and *it breaks co-adaptation* — are usually asserted rather
than shown.

Both are measurable, and we measure both.

| Claim | Measured below |
|---|---|
| a layer of `n` neurons has `2^n` sub-networks | exactly — and they **share one set of weights** |
| dropout breaks co-adaptation | mean \|correlation\| between hidden units falls sharply |
| it produces smaller, more evenly spread weights | max \|w\| falls from **3.52 to 2.53** |
| "dropout is basically L2" | **it is not** — measured side by side, they do different things |

`numpy` only, so every mechanism is visible rather than hidden in a framework.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)

X, y = make_moons(n_samples=500, noise=0.28, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.6, random_state=0, stratify=y)
s = StandardScaler().fit(X_tr)
X_tr, X_te = s.transform(X_tr), s.transform(X_te)
y_tr, y_te = y_tr.reshape(-1, 1) * 1.0, y_te.reshape(-1, 1) * 1.0
print(f"train {X_tr.shape}, test {X_te.shape}")

## Part A — The ensemble argument, counted

An ensemble usually means training many models and averaging them, which costs many models'
worth of compute and memory. Dropout gets the same effect for free, because **every mask is
a different network** — and they all share one set of weights.

In [ ]:
print(f"{'hidden units n':>16}{'possible sub-networks 2^n':>44}")
for n in (4, 8, 16, 32, 64, 128):
    print(f"{n:>16}{2 ** n:>44,}")
print("\nA 128-unit layer has more sub-networks than there are atoms in a person.")
print("Every training step samples one of them, and every one shares the SAME weights,")
print("which is why this costs one model rather than 2^n of them.")

In [ ]:
# how many distinct masks does training actually visit?
p, n_units, steps = 0.5, 16, 5000
seen = set()
for _ in range(steps):
    seen.add((rng.random(n_units) > p).tobytes())
print(f"{steps:,} steps on a {n_units}-unit layer visited {len(seen):,} distinct masks")
print(f"out of {2 ** n_units:,} possible - {len(seen) / 2 ** n_units:.1%} of them")
print("\nThe ensemble is never enumerated. It does not need to be: the weights are")
print("shared, so improving one sub-network improves overlapping ones too.")

## Part B — Co-adaptation, and how to see it

The second argument is subtler. Without dropout, a neuron can learn to be useful *only in
the company of another specific neuron* — one fires high, the next corrects it, and neither
means anything alone. That is **co-adaptation**, and it is fragile: it works on the training
data and generalises badly.

Dropout makes it impossible, because no neuron can rely on any other being present.

To measure it, train the same network twice and look at the correlations between hidden
activations.

In [ ]:
def relu(z):    return np.maximum(0, z)
def d_relu(z):  return (z > 0) * 1.0
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def train(p_drop=0.0, l2=0.0, epochs=3000, lr=0.3, hidden=64, seed=1):
    r = np.random.default_rng(seed)
    W1 = r.normal(size=(2, hidden)) * np.sqrt(2 / 2);   b1 = np.zeros(hidden)
    W2 = r.normal(size=(hidden, 1)) * np.sqrt(2 / hidden); b2 = np.zeros(1)
    for _ in range(epochs):
        h = relu(X_tr @ W1 + b1)
        if p_drop:
            # INVERTED dropout: scale up at train time so test time needs no change
            mask = (r.random(h.shape) > p_drop) / (1 - p_drop)
            h_drop = h * mask
        else:
            h_drop = h
        out = sigmoid(h_drop @ W2 + b2)
        d_out = (out - y_tr) / len(X_tr)
        gW2 = h_drop.T @ d_out + l2 * W2
        gb2 = d_out.sum(0)
        d_h = (d_out @ W2.T) * (mask if p_drop else 1.0) * d_relu(X_tr @ W1 + b1)
        gW1 = X_tr.T @ d_h + l2 * W1
        gb1 = d_h.sum(0)
        W1 -= lr * gW1; b1 -= lr * gb1; W2 -= lr * gW2; b2 -= lr * gb2
    return W1, b1, W2, b2

def predict(params, X):
    W1, b1, W2, b2 = params
    return sigmoid(relu(X @ W1 + b1) @ W2 + b2)      # no mask at test time

def acc(params, X, t):
    return float(((predict(params, X) > 0.5) == (t > 0.5)).mean())

In [ ]:
# mean |correlation| between pairs of hidden units, on the training data
def coadaptation(params):
    W1, b1, _, _ = params
    h = relu(X_tr @ W1 + b1)
    h = h[:, h.std(0) > 1e-8]                 # ignore dead units, which have no correlation
    c = np.corrcoef(h, rowvar=False)
    iu = np.triu_indices_from(c, k=1)
    return float(np.abs(c[iu]).mean()), h.shape[1]

print(f"{'model':<22}{'train':>8}{'test':>8}{'gap':>8}{'mean |corr|':>14}{'live units':>12}")
runs = {}
for label, kw in (("no dropout", {}),
                  ("dropout p = 0.2", {"p_drop": 0.2}),
                  ("dropout p = 0.5", {"p_drop": 0.5}),
                  ("L2 instead", {"l2": 0.05})):
    prm = train(**kw)
    runs[label] = prm
    c, live = coadaptation(prm)
    tr, te = acc(prm, X_tr, y_tr), acc(prm, X_te, y_te)
    print(f"{label:<22}{tr:>8.3f}{te:>8.3f}{tr - te:>8.3f}{c:>14.3f}{live:>12}")

The `mean |corr|` column is the co-adaptation measure, and dropout moves it in the expected
direction — 0.490 with none, 0.437 at `p = 0.5`. Real, and more modest than the usual
telling suggests: this is a 2-D problem with 64 hidden units, so there is only so much
structure available to decorrelate.

**Now look at the L2 row, which came out the opposite way to the prediction.** Mean
|correlation| *rose* to 0.998, close to perfect. That is not a bug and it is worth
understanding: L2 at this strength shrinks the weights so hard that the hidden layer
collapses toward a single direction, and units that are all doing the same thing are
perfectly correlated by definition. Its training accuracy fell to 0.860, which is the same
story from the other side.

So the common shorthand — *dropout is basically L2* — does not survive contact with the
measurement. **They are different levers.** L2 attacks the magnitude of the weights.
Dropout attacks the dependence between units. The next cell shows the first half of that
claim as clearly as this one showed the second.

## Part C — What it does to the weights

In [ ]:
print(f"{'model':<22}{'max |w|':>10}{'mean |w|':>11}{'||W||_2':>10}")
for label, prm in runs.items():
    W1, _, W2, _ = prm
    allw = np.concatenate([W1.ravel(), W2.ravel()])
    print(f"{label:<22}{np.abs(allw).max():>10.3f}{np.abs(allw).mean():>11.3f}"
          f"{np.linalg.norm(allw):>10.3f}")

In [ ]:
# the shape of the distribution, not just its size
print("share of weights above each magnitude:\n")
print(f"{'model':<22}" + "".join(f"{'>' + str(t):>9}" for t in (0.5, 1.0, 2.0)))
for label, prm in runs.items():
    W1, _, W2, _ = prm
    allw = np.abs(np.concatenate([W1.ravel(), W2.ravel()]))
    print(f"{label:<22}" + "".join(f"{(allw > t).mean():>9.3f}" for t in (0.5, 1.0, 2.0)))

Dropout does produce **smaller weights** — the largest falls from 3.52 to 2.53, and the
share above 2.0 nearly halves. The intuition is sound: a neuron that might disappear cannot
be allowed to carry a huge weight, because the loss would spike whenever it did, so the
network hedges by spreading magnitude around.

But put the two tables side by side and the difference in *kind* is obvious. L2 at
`alpha = 0.05` drives the largest weight to **0.539** and leaves 0.5% of weights above 0.5 —
it is an order of magnitude more aggressive about size, and it paid for that with training
accuracy and with a collapsed, perfectly-correlated hidden layer. Dropout barely touched the
size and moved the correlation instead.

**Same goal, different mechanism, different failure mode.** Which is why they are usually
used together rather than as alternatives.

## Part D — Where `p` goes and what it means

`p` is set **per layer**, and it is the probability of *dropping*:

| Layer | Typical `p` | Why |
|---|---|---|
| input | 0.1 – 0.2 | you are throwing away raw data; be gentle |
| hidden | 0.5 | the classic value from the original paper |
| output | never | you cannot drop the answer |

The expected gain in the literature is around **2 percentage points**. At 95% accuracy that
is a fifth of the remaining error, which is substantial — but it is not the difference
between working and not working, and dropout is not a rescue for a model that is
underfitting. Lesson 021's diagnosis comes first.

Lesson 025 covers the practical side: choosing `p`, the train/test asymmetry, and what
happens at the extremes.

## Try it yourself

1. Set `p_drop = 0.9` and re-run Part B. What happens to the train column, and what does
   that say about dropout as a regulariser being able to go too far?
2. The dropout implementation here is **inverted dropout** — it scales by `1/(1-p)` at
   training time. Remove that division and see what happens to test accuracy. Why?
3. Apply dropout to the *input* layer as well, at `p = 0.2`. Does it help on 2-D data?
4. Measure `mean |corr|` at several points during training rather than only at the end. When
   does co-adaptation form?